In [4]:
# Compute the mean ElderReact embedding (mu_source) with the fine-tuned model and save it.
# Embedding = the input of model.head, preprocessed exactly like infer_long_videos.py (16 of the 32 cached frames, ImageNet norm).
#
# Run:  python compute_mu_source.py
# Out:  models/mu_source.npy  (512,)   + models/mu_source_meta.txt

import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
import sys, glob
import numpy as np
import torch

# ============================ CONFIG ============================
REPO_DIR  = 'MAE-DFER'
CKPT      = 'models/MAE_DFER-ElderReact.pth'
ER_CACHE  = 'cache/er_cache'                      # folder with the ElderReact .npy clips (searched recursively)
OUT_FILE_MU  = 'models/mu_source.npy'
OUT_FILE_STD = 'models/std_source.npy'
# Optional: only average the clips listed in this label file (e.g. the TRAIN split). None = every .npy in ER_CACHE.
LABEL_FILE = None                           # e.g. 'labels/train_labels.txt'
BATCH_SIZE = 8
DEVICE = 'auto'

# ---- must match training / inference ----
MODEL_NAME = 'vit_base_dim512_no_depth_patch16_160'
DEPTH, ATTN_TYPE, LG_REGION_SIZE, LG_CLASSIFY_TOKEN_TYPE = 16, 'local_global', [2, 5, 10], 'region'
NUM_CLASSES = 6                             # ElderReact head
NUM_FRAMES, TUBELET = 16, 2
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
# =================================================================


def clip_paths():
    paths = [p for p in glob.glob(os.path.join(ER_CACHE, '**', '*.npy'), recursive=True) if not p.endswith('.tmp.npy')]
    if LABEL_FILE:
        stems = set()
        with open(LABEL_FILE) as f:
            for line in f:
                parts = line.replace(',', ' ').split()
                if len(parts) >= 7:
                    name = os.path.basename(parts[0])
                    stems.update({name, os.path.splitext(name)[0]})
        # cache names are '<stem>_<8 hex>.npy'; match on the stem
        paths = [p for p in paths if os.path.basename(p)[:-4].rsplit('_', 1)[0] in stems]
    return sorted(paths)


def load_clip(path):
    arr = np.load(path)                                           # K,160,160,3 uint8
    per = max(1, len(arr) // NUM_FRAMES)
    x = (arr[[s * per for s in range(NUM_FRAMES)]].astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(np.ascontiguousarray(x.transpose(3, 0, 1, 2)))     # C,T,H,W


def main():
    device = ('mps' if torch.backends.mps.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE
    sys.path.append(REPO_DIR)
    import modeling_finetune  # noqa: F401
    from timm.models import create_model
    model = create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES, all_frames=NUM_FRAMES,
                         tubelet_size=TUBELET, drop_path_rate=0.0, depth=DEPTH, attn_type=ATTN_TYPE,
                         lg_region_size=LG_REGION_SIZE, lg_classify_token_type=LG_CLASSIFY_TOKEN_TYPE)
    ck = torch.load(CKPT, map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'] if 'model' in ck else ck, strict=True)
    model = model.to(device).eval()

    paths = clip_paths()
    if not paths:
        raise SystemExit(f'no clips found in {ER_CACHE} (LABEL_FILE={LABEL_FILE})')
    print(f'device: {device} | {len(paths)} clips')

    feats = []
    handle = model.head.register_forward_pre_hook(lambda m, inp: feats.append(inp[0].detach().float().cpu()))
    with torch.no_grad():
        for i in range(0, len(paths), BATCH_SIZE):
            model(torch.stack([load_clip(p) for p in paths[i:i + BATCH_SIZE]]).to(device))
            if (i // BATCH_SIZE) % 20 == 0:
                print(f'   {min(i + BATCH_SIZE, len(paths))}/{len(paths)}', flush=True)
    handle.remove()

    F = torch.cat(feats).numpy().astype(np.float64)
    mu = F.mean(0).astype(np.float32)
    std = F.std(0).astype(np.float32)
    os.makedirs(os.path.dirname(OUT_FILE_MU) or '.', exist_ok=True)
    os.makedirs(os.path.dirname(OUT_FILE_STD) or '.', exist_ok=True)
    np.save(OUT_FILE_MU, mu)
    np.save(OUT_FILE_STD, std)
            # with open(OUT_FILE.replace('.npy', '_meta.txt'), 'w') as f:
            #     f.write(f'ckpt={CKPT}\nclips={len(F)}\nlabel_file={LABEL_FILE}\ndim={mu.shape[0]}\n'
            #             f'norm={np.linalg.norm(mu):.4f}\nmean_per_dim_std={F.std(0).mean():.4f}\n')
    print(f'saved {OUT_FILE_MU} and {OUT_FILE_STD}: shape {mu.shape}, ||mu||={np.linalg.norm(mu):.3f}, '
          f'mean per-dim std across clips={F.std(0).mean():.3f}')


if __name__ == '__main__':
    main()

==> Note: Use 'local_global' for compute reduction (lg_region_size=[2, 5, 10],lg_first_attn_type=self, lg_third_attn_type=cross,lg_attn_param_sharing_first_third=False,lg_attn_param_sharing_all=False,lg_classify_token_type=region,lg_no_second=False, lg_no_third=False)
==> Number of local regions: 8 (size=[4, 2, 1])
device: mps | 1323 clips
   8/1323
   168/1323
   328/1323
   488/1323
   648/1323
   808/1323
   968/1323
   1128/1323
   1288/1323
saved models/mu_source.npy and models/std_source.npy: shape (512,), ||mu||=9.714, mean per-dim std across clips=0.713
